# Lab 01.3 — Calcolo parallelo e distribuito

## Introduzione
Esempi di problemi **embarrassingly parallel** (indipendenti tra loro) e degli strumenti che Spark offre per il calcolo distribuito.

### Obiettivi
1. Inizializzare la `SparkSession` e leggere i core disponibili.
2. Stima Monte Carlo di π e **scalabilità** al variare delle partizioni.
3. Calcolo CPU-intensive distribuito (numeri primi).
4. `mapPartitions`: inizializzazione costosa una volta per partizione.
5. Variabili condivise: **broadcast** e **accumulatori**.
6. **Caching** dei dati riutilizzati.
7. Tolleranza ai guasti: ricalcolo dal lineage.
8. Chiudere la sessione.

## Step 1: Inizializzare la SparkSession
Creiamo la sessione con il master definito da `SPARK_MASTER` (cluster Standalone; in assenza, `local[*]`) e memorizziamo in `CORES` il parallelismo di default, cioè i core totali disponibili: lo useremo per scegliere il numero di partizioni.

In [ ]:
import os, time, random, math, socket
from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("Lab01-CalcoloParallelo")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .getOrCreate())
sc = spark.sparkContext
# Core totali disponibili: base per scegliere il numero di partizioni
CORES = sc.defaultParallelism
print("Master:", sc.master, "| core disponibili:", CORES)

## Step 2: Monte Carlo, stima di π
Lanciamo N punti casuali nel quadrato [0,1]²: la frazione che cade nel quarto di cerchio vale π/4.
Ogni partizione genera i propri punti **senza comunicare** con le altre; alla fine un `sum` aggrega i risultati.
Ogni task riceve un seme diverso (`seed`) per generare sequenze indipendenti e riproducibili.

In [ ]:
def punti_nel_cerchio(seed_and_n):
    seed, n = seed_and_n
    rnd = random.Random(seed)       # generatore con seme proprio: nessuna comunicazione tra task
    dentro = 0
    for _ in range(n):
        x, y = rnd.random(), rnd.random()
        if x * x + y * y <= 1.0:    # il punto cade nel quarto di cerchio
            dentro += 1
    return dentro

def stima_pi(n_totale, n_partizioni):
    per_part = n_totale // n_partizioni
    # Un elemento (seed, n) per partizione: ogni task lavora in autonomia
    tasks = sc.parallelize([(seed, per_part) for seed in range(n_partizioni)], n_partizioni)
    dentro = tasks.map(punti_nel_cerchio).sum()   # unica fase di aggregazione
    return 4.0 * dentro / (per_part * n_partizioni)

N = 10_000_000
t0 = time.time()
pi = stima_pi(N, CORES)
print(f"π ≈ {pi:.6f}  (errore {abs(pi - math.pi):.6f})  con {N:,} punti in {time.time() - t0:.2f}s")

### Step 2b: Scalabilità
Con 1 partizione lavora un solo core; aumentando le partizioni il lavoro si divide tra gli executor.
Oltre il numero di core il tempo non scende più (e piccoli task aggiungono overhead di scheduling).
Ripetiamo la stima con 1, 2, 4, 8, 16 e 64 partizioni e calcoliamo lo **speedup** rispetto a una sola partizione.

In [ ]:
risultati = []
for p in [1, 2, 4, 8, 16, 64]:
    t0 = time.time()
    stima_pi(N, p)
    risultati.append((p, time.time() - t0))

# Speedup = tempo con 1 partizione / tempo con p partizioni
base = risultati[0][1]
print("partizioni   tempo(s)   speedup")
for p, t in risultati:
    print(f"{p:10d} {t:10.2f} {base / t:9.2f}x  {'█' * int(10 * base / t)}")

### Step 2c: Grafico dello speedup
Rappresentiamo lo speedup misurato e lo confrontiamo con il numero di core disponibili (linea tratteggiata): è il limite teorico oltre il quale più partizioni non aiutano.

In [ ]:
import matplotlib.pyplot as plt
ps, ts = zip(*risultati)
plt.figure(figsize=(6, 3))
plt.plot(ps, [base / t for t in ts], "o-", label="speedup misurato")
# Linea orizzontale = limite teorico (numero di core)
plt.axhline(CORES, color="gray", ls="--", label=f"core disponibili ({CORES})")
plt.xscale("log", base=2); plt.xlabel("partizioni"); plt.ylabel("speedup"); plt.legend(); plt.grid(alpha=.3)
plt.title("Monte Carlo π: speedup al variare delle partizioni"); plt.show()

### Punti chiave (Monte Carlo)
* Un problema embarrassingly parallel scala bene: i task non comunicano e si aggrega solo alla fine.
* Lo speedup cresce fino al numero di core, poi si stabilizza; troppe partizioni aggiungono solo overhead.

## Step 3: Calcolo CPU-intensive, quanti numeri primi sotto 2 milioni?
Ogni partizione riceve un **intervallo** di numeri e lo verifica in autonomia.
Dividiamo l'intervallo [0, 2 milioni) in 16 blocchi, uno per partizione; ogni task restituisce anche il nome dell'host, per vedere come il lavoro si ripartisce tra gli executor.

In [ ]:
def primi_in(intervallo):
    a, b = intervallo
    def primo(n):
        # Test di primalità per divisione: si provano solo i divisori dispari fino a sqrt(n)
        if n < 2: return False
        if n % 2 == 0: return n == 2
        r = int(n ** 0.5)
        for d in range(3, r + 1, 2):
            if n % d == 0: return False
        return True
    # (host che ha eseguito il task, quantità di primi trovati nel blocco)
    return (socket.gethostname(), sum(1 for n in range(a, b) if primo(n)))

LIMITE, BLOCCHI = 2_000_000, 16
passo = LIMITE // BLOCCHI
intervalli = [(i * passo, (i + 1) * passo) for i in range(BLOCCHI)]

t0 = time.time()
parziali = sc.parallelize(intervalli, BLOCCHI).map(primi_in).collect()
print(f"Numeri primi < {LIMITE:,}: {sum(c for _, c in parziali):,}  in {time.time() - t0:.2f}s")
for host in sorted({h for h, _ in parziali}):
    print(f"  {host}: {sum(1 for h, _ in parziali if h == host)} blocchi elaborati")

### Punti chiave (CPU-intensive)
* Dividere il dominio in intervalli indipendenti permette di sfruttare tutti i core del cluster.
* Restituire l'host nei risultati è un modo semplice per verificare la distribuzione del lavoro.

## Step 4: `map` vs `mapPartitions`
Se ogni elemento richiede una risorsa costosa (connessione a DB, modello ML, ...), con `mapPartitions` la si crea **una volta per partizione** invece che una volta per elemento.
Simuliamo il costo con uno `sleep` di 10 ms e confrontiamo i tempi su 400 elementi in 4 partizioni (400 inizializzazioni contro 4).

In [ ]:
def risorsa_costosa():
    time.sleep(0.01)          # simula l'apertura di una connessione
    return lambda x: x * 2

dati = sc.parallelize(range(400), 4)

# map: la risorsa viene creata per ogni elemento
t0 = time.time()
dati.map(lambda x: risorsa_costosa()(x)).count()
t_map = time.time() - t0

# mapPartitions: la risorsa viene creata una volta e riusata per tutta la partizione
def per_partizione(it):
    f = risorsa_costosa()     # 1 volta per partizione
    for x in it:
        yield f(x)

t0 = time.time()
dati.mapPartitions(per_partizione).count()
t_mp = time.time() - t0
print(f"map: {t_map:.2f}s  (400 inizializzazioni)   mapPartitions: {t_mp:.2f}s  (4 inizializzazioni)")

### Punti chiave (mapPartitions)
* `mapPartitions` ammortizza i costi di inizializzazione su tutti gli elementi di una partizione.
* È la scelta giusta per connessioni, client o modelli da caricare.

## Step 5: Variabili condivise, broadcast e accumulatori
* **Broadcast**: dato di sola lettura inviato **una volta per executor** (non una volta per task).
* **Accumulatore**: contatore in sola scrittura per i task, leggibile dal driver.

Usiamo un dizionario di capitali come broadcast e un accumulatore che conta i codici non riconosciuti.

In [ ]:
# Dizionario di lookup inviato a tutti gli executor
capitali = sc.broadcast({"IT": "Roma", "FR": "Parigi", "DE": "Berlino", "ES": "Madrid"})
# Contatore condiviso: i task possono solo incrementarlo
codici_sconosciuti = sc.accumulator(0)

def risolvi(codice):
    if codice not in capitali.value:
        codici_sconosciuti.add(1)
        return (codice, None)
    return (codice, capitali.value[codice])

codici = sc.parallelize(["IT", "FR", "XX", "DE", "IT", "YY", "ES"] * 1000, 8)
print(codici.map(risolvi).distinct().collect())
# Il valore si legge sul driver dopo l'azione
print("Codici non riconosciuti (accumulatore):", codici_sconosciuti.value)

### Punti chiave (variabili condivise)
* Il broadcast evita di rispedire un dato grande a ogni task.
* Gli accumulatori vanno letti solo sul driver, dopo un'azione; nelle trasformazioni possono essere aggiornati più volte se un task viene rieseguito.

## Step 6: Caching
Senza cache ogni azione ricalcola l'RDD dall'inizio del lineage. Con `cache()`/`persist()` le partizioni restano in memoria negli executor (scheda *Storage* della Spark UI).
Misuriamo due `sum()` senza cache e tre con `persist(MEMORY_ONLY)`: la prima popola la cache, le successive la leggono dalla RAM e sono più veloci.

In [ ]:
# RDD costoso da calcolare (funzioni matematiche su 2 milioni di elementi)
costoso = sc.parallelize(range(2_000_000), 8).map(lambda x: math.sqrt(x) * math.sin(x))

# Senza cache: ogni azione ricalcola tutto il lineage
for etichetta in ["senza cache", "senza cache"]:
    t0 = time.time(); costoso.sum(); print(f"{etichetta:12s}: {time.time() - t0:.2f}s")

# Con cache: la prima azione popola la memoria, le successive la riusano
costoso.persist(StorageLevel.MEMORY_ONLY)
for etichetta in ["1ª (popola)", "2ª (in RAM)", "3ª (in RAM)"]:
    t0 = time.time(); costoso.sum(); print(f"{etichetta:12s}: {time.time() - t0:.2f}s")
costoso.unpersist()

### Punti chiave (caching)
* La cache conviene quando lo stesso RDD viene usato da più azioni.
* La prima azione paga il costo di popolamento; le successive leggono dalla memoria.

## Step 7: Tolleranza ai guasti
Se un task fallisce, Spark lo **riesegue** (anche su un altro executor) ricostruendo la partizione dal lineage.
Simuliamo un errore al primo tentativo di una partizione: il job termina comunque correttamente.

> In modalità `local[*]` i tentativi sono disabilitati di default (`local[N, maxFailures]`): l'esempio mostra la differenza.

In [ ]:
from pyspark import TaskContext

def fallisce_al_primo_tentativo(it):
    ctx = TaskContext.get()
    # Solo la partizione 0, solo al primo tentativo: guasto simulato
    if ctx.partitionId() == 0 and ctx.attemptNumber() == 0:
        raise RuntimeError("guasto simulato!")
    return [sum(it)]

try:
    print("Somma:", sum(sc.parallelize(range(100), 4).mapPartitions(fallisce_al_primo_tentativo).collect()))
    print("Il task fallito è stato rieseguito: vedi 'Failed Tasks' nella scheda Stages.")
except Exception as e:
    print("Job fallito (nessun nuovo tentativo in local mode):", type(e).__name__)

### Punti chiave (tolleranza ai guasti)
* Spark riesegue i task falliti usando il lineage: il risultato finale resta corretto.
* Il numero di tentativi dipende dalla modalità: in `local[*]` il job fallisce al primo errore.

## Step 8: Chiudere la sessione
Terminiamo l'applicazione per liberare le risorse del cluster.

In [ ]:
# Rilascia gli executor sul cluster
spark.stop()

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Parallelismo | `sc.defaultParallelism`, `sc.parallelize(dati, n)` | sceglie quante partizioni (task) usare |
| Embarrassingly parallel | Monte Carlo, intervalli di primi + `sum`/`collect` | task indipendenti, aggregazione finale |
| Scalabilità | speedup = t(1 partizione) / t(p partizioni) | misura quanto il calcolo scala sui core |
| Costo di inizializzazione | `mapPartitions` | crea una risorsa costosa una volta per partizione |
| Broadcast | `sc.broadcast(dati)` | invia un dato di sola lettura una volta per executor |
| Accumulatore | `sc.accumulator(0)` | contatore scritto dai task, letto dal driver |
| Caching | `persist(StorageLevel.MEMORY_ONLY)`, `unpersist()` | evita di ricalcolare un RDD riusato |
| Tolleranza ai guasti | lineage, `TaskContext.attemptNumber()` | riesegue i task falliti ricostruendo le partizioni |
| Chiusura | `spark.stop()` | libera gli executor sul cluster |